# Generate supervised speech-denoising datasets

Uses the supplied Common Voice splits and separate UrbanSound8K folds.
Defaults: 7,000 train / 1,000 validation / 1,000 test pairs at 44.1 kHz.
Noise can repeat within a split. These are fixed supervised noisy/clean pairs, not Noise2Noise pairs.
Use a fresh output directory for each generation attempt; incomplete runs have no COMPLETE.json.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xf /content/drive/MyDrive/Denoiser/CommonVoice27_PL.tar -C /content/
!tar -xf /content/drive/MyDrive/Denoiser/UrbanSound8K.tar -C /content/

In [ ]:
!pip install torchaudio soundfile matplotlib tqdm


In [ ]:
import os
import random
import torch
import torchaudio
import soundfile as sf
import numpy as np
from pathlib import Path

In [ ]:

def generate_noisy_speech(
    speech_path,
    noise_path,
    output_dir,
    sample_rate=16000,
    snr_db=5,
    peak_level=0.99,
    filename="sample",
    seed=None
):
    """
    Generate a paired noisy/clean audio example.

    Args:
        speech_path: Path to clean speech WAV/audio file.
        noise_path: Path to environmental noise WAV/audio file.
        output_dir: Directory where output files will be saved.
        sample_rate: Target sample rate (default: 16000 Hz).
        snr_db: Desired signal-to-noise ratio in dB.
        peak_level: Maximum absolute amplitude after normalization.
        filename: Output filename without extension.
        seed: Optional random seed for noise cropping.

    Returns:
        noisy_path, clean_path
    """

    # 1. Load audio
    speech, sr_speech = torchaudio.load(speech_path)
    noise, sr_noise = torchaudio.load(noise_path)

    # 2. Convert stereo/multichannel audio to mono
    speech = speech.mean(dim=0, keepdim=True)
    noise = noise.mean(dim=0, keepdim=True)

    # 3. Resample to the same sample rate
    if sr_speech != sample_rate:
        speech = torchaudio.functional.resample(
            speech, sr_speech, sample_rate
        )

    if sr_noise != sample_rate:
        noise = torchaudio.functional.resample(
            noise, sr_noise, sample_rate
        )

    # 4. Check speech length
    speech_length = speech.shape[1]

    if speech_length == 0:
        raise ValueError("Speech recording is empty.")

    if torch.sqrt(torch.mean(speech ** 2)) < 1e-8:
        raise ValueError("Speech recording is silent.")

    # 5. Make noise exactly the same length as speech
    noise_length = noise.shape[1]

    if noise_length == 0:
        raise ValueError("Noise recording is empty.")

    if noise_length < speech_length:

        # Repeat noise if it is shorter than speech
        repetitions = int(np.ceil(speech_length / noise_length))
        noise = noise.repeat(1, repetitions)

    if noise.shape[1] > speech_length:

        # Select a random segment of the required length
        max_start = noise.shape[1] - speech_length

        rng = random.Random(seed)
        start = rng.randint(0, max_start)

        noise = noise[:, start:start + speech_length]

    # 6. Verify that both signals have identical dimensions
    assert speech.shape == noise.shape, "Audio lengths do not match!"

    # 7. Adjust noise according to the desired SNR
    speech_rms = torch.sqrt(torch.mean(speech ** 2))
    noise_rms = torch.sqrt(torch.mean(noise ** 2))

    if noise_rms < 1e-8:
        raise ValueError("Noise recording is silent.")

    snr_linear = 10 ** (snr_db / 20)

    noise = noise * (speech_rms / (snr_linear * noise_rms))

    # 8. Mix speech and noise
    noisy = speech + noise

    # 9. Use the same scale, protecting BOTH clean and noisy from clipping
    peak = torch.maximum(noisy.abs().max(), speech.abs().max())

    if peak > 0:
        scale = peak_level / peak
        noisy = noisy * scale
        clean = speech * scale
    else:
        clean = speech.clone()

    # 10. Verify lengths and finite values
    assert noisy.shape == clean.shape
    assert torch.isfinite(noisy).all()
    assert torch.isfinite(clean).all()

    # 11. Save audio files
    os.makedirs(output_dir, exist_ok=True)

    noisy_path = os.path.join(output_dir, f"{filename}_noisy.wav")
    clean_path = os.path.join(output_dir, f"{filename}_clean.wav")

    sf.write(
        noisy_path,
        noisy.squeeze(0).numpy(),
        sample_rate,
        subtype="PCM_16"
    )

    sf.write(
        clean_path,
        clean.squeeze(0).numpy(),
        sample_rate,
        subtype="PCM_16"
    )

    return noisy_path, clean_path


In [ ]:
noisy_path, clean_path = generate_noisy_speech(
    speech_path="/content/CommonVoice27_PL/pl/clips/common_voice_pl_20554701.mp3",
    noise_path="/content/UrbanSound8K/audio/fold1/7061-6-0-0.wav",
    output_dir="/content/denoiser_dataset/train",
    sample_rate=44100,
    snr_db=5,
    filename="example_001",
    seed=42
)

In [ ]:
import IPython.display as ipd

print("Noisy speech:")
display(ipd.Audio(noisy_path))

print("Clean speech:")
display(ipd.Audio(clean_path))

In [ ]:
import csv
import json
from itertools import combinations
from collections import Counter

SEED = 6767
SAMPLE_RATE = 44100
SNR_LEVELS = [-5, 0, 5, 10]
COUNTS = {"train": 7000, "val": 1000, "test": 1000}
CV_ROOT = Path("/content/CommonVoice27_PL/pl")
NOISE_ROOT = Path("/content/UrbanSound8K")
OUTPUT_DIR = Path("/content/DenoiserData_v2")
CV_FILES = {"train": "train.tsv", "val": "dev.tsv", "test": "test.tsv"}
NOISE_FOLDS = {"train": list(range(1, 9)), "val": [9], "test": [10]}
FOLDERS = {"train": "Train", "val": "Validation", "test": "Test"}


def read_rows(path, tsv=False):
    with open(path, encoding="utf-8", newline="") as f:
        # Common Voice text may contain literal quotes: do not interpret them
        # as CSV quoting, which can merge rows and silently drop clips.
        return list(csv.DictReader(
            f, delimiter="\t" if tsv else ",",
            quoting=csv.QUOTE_NONE if tsv else csv.QUOTE_MINIMAL,
        ))


def require(condition, message):
    if not condition:
        raise ValueError(message)


def check_disjoint(groups, key):
    for a, b in combinations(groups, 2):
        overlap = {r[key] for r in groups[a]} & {r[key] for r in groups[b]}
        require(not overlap, f"{key} overlap between {a} and {b}: {len(overlap)}")


speech_pools = {}
for split, filename in CV_FILES.items():
    rows = read_rows(CV_ROOT / filename, tsv=True)
    require(rows and all(r.get("client_id") and r.get("path") for r in rows),
            f"Missing speech filenames or contributor IDs: {filename}")
    speech_pools[split] = [
        {"client_id": r["client_id"], "speech_path": str(CV_ROOT / "clips" / r["path"])}
        for r in rows
    ]
    paths = [r["speech_path"] for r in speech_pools[split]]
    require(len(paths) == len(set(paths)), f"Duplicate speech files: {split}")
    require(len(rows) >= COUNTS[split], f"Not enough speech for {split}")

noise_rows = read_rows(NOISE_ROOT / "metadata" / "UrbanSound8K.csv")
# Some original recordings span folds. Assign each source using its lowest
# fold number, then keep ALL of its excerpts in that source's split.
source_fold = {}
for r in noise_rows:
    source_fold[r["fsID"]] = min(source_fold.get(r["fsID"], 11), int(r["fold"]))
noise_pools = {}
for split, folds in NOISE_FOLDS.items():
    noise_pools[split] = [
        {"noise_path": str(NOISE_ROOT / "audio" / f"fold{r['fold']}" / r["slice_file_name"]),
         "noise_source_id": r["fsID"], "noise_fold": int(r["fold"]),
         "noise_class": int(r["classID"]), "noise_split_fold": source_fold[r["fsID"]]}
        for r in noise_rows if source_fold[r["fsID"]] in folds
    ]
    require({r["noise_class"] for r in noise_pools[split]} == set(range(10)),
            f"Expected all 10 noise classes in {split}")

check_disjoint(speech_pools, "client_id")
check_disjoint(speech_pools, "speech_path")
check_disjoint(noise_pools, "noise_source_id")
check_disjoint(noise_pools, "noise_path")
print("Source pools checked: no contributor or original noise recording overlaps.")
for split in COUNTS:
    print(split, "speech:", len(speech_pools[split]), "noise:", len(noise_pools[split]))


In [ ]:
from tqdm.auto import tqdm


def write_rows(path, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


# Plan all pairs before writing audio. Local RNGs make reruns reproducible.
plans = {}
for split_index, (split, count) in enumerate(COUNTS.items()):
    require(count > 0, "Pair counts must be positive")
    split_seed = SEED + split_index * 1_000_000
    rng = random.Random(split_seed)
    speech = rng.sample(speech_pools[split], count)
    classes = [i % 10 for i in range(count)]
    snrs = [SNR_LEVELS[i % len(SNR_LEVELS)] for i in range(count)]
    rng.shuffle(classes)
    rng.shuffle(snrs)
    by_class = {c: [r for r in noise_pools[split] if r["noise_class"] == c]
                for c in range(10)}
    plans[split] = []
    for i, (voice, noise_class, snr) in enumerate(zip(speech, classes, snrs)):
        noise = rng.choice(by_class[noise_class])  # reuse allowed within split
        name = f"{split}_{i:05d}"
        folder = FOLDERS[split]
        plans[split].append({
            "id": name, "split": split, **voice, **noise,
            "noisy_path": f"{folder}/{name}_noisy.wav",
            "clean_path": f"{folder}/{name}_clean.wav",
            "snr_db": snr, "crop_seed": split_seed + i,
            "sample_rate": SAMPLE_RATE,
        })

# Verify selected source files before a long generation run.
for split, rows in plans.items():
    for r in rows:
        for key in ("speech_path", "noise_path"):
            require(Path(r[key]).is_file(), f"Missing input: {r[key]}")
for key in ("client_id", "speech_path", "noise_source_id", "noise_path"):
    check_disjoint(plans, key)

require(not OUTPUT_DIR.exists() or not any(OUTPUT_DIR.iterdir()),
        f"{OUTPUT_DIR} is not empty. Use a fresh directory; do not mix old and new datasets.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = {"seed": SEED, "sample_rate": SAMPLE_RATE, "counts": COUNTS,
          "snr_levels": SNR_LEVELS, "common_voice_splits": CV_FILES,
          "noise_folds": NOISE_FOLDS, "cross_fold_source_policy": "assign by lowest original fold",
          "noise_sampling": "balanced classes, sampling with replacement within each split",
          "output_paths": "relative to dataset root",
          "versions": {"torch": torch.__version__, "torchaudio": torchaudio.__version__,
                       "numpy": np.__version__, "soundfile": sf.__version__}}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

for split, rows in plans.items():
    write_rows(OUTPUT_DIR / f"{split}_plan.csv", rows)
    for r in tqdm(rows, desc=f"Generating {split}"):
        # Fail immediately on bad inputs, rather than silently losing examples.
        generate_noisy_speech(
            speech_path=r["speech_path"], noise_path=r["noise_path"],
            output_dir=OUTPUT_DIR / FOLDERS[split],
            sample_rate=SAMPLE_RATE, snr_db=r["snr_db"],
            filename=r["id"], seed=r["crop_seed"],
        )
        clean, clean_sr = sf.read(OUTPUT_DIR / r["clean_path"], dtype="float64")
        noisy, noisy_sr = sf.read(OUTPUT_DIR / r["noisy_path"], dtype="float64")
        require(clean_sr == noisy_sr == SAMPLE_RATE, "Wrong sample rate")
        require(clean.ndim == noisy.ndim == 1 and clean.shape == noisy.shape
                and clean.size > 0, "Invalid audio dimensions")
        require(np.isfinite(clean).all() and np.isfinite(noisy).all(), "Nonfinite audio")
        require(max(np.abs(clean).max(), np.abs(noisy).max()) < 1.0, "Clipped audio")
        speech_power = np.mean(clean ** 2)
        noise_power = np.mean((noisy - clean) ** 2)
        require(speech_power > 0 and noise_power > 0, "Silent speech or noise")
        measured_snr = float(10 * np.log10(speech_power / noise_power))
        require(abs(measured_snr - r["snr_db"]) < 0.1,
                f"Saved SNR mismatch: {r['id']} ({measured_snr})")
        r["measured_snr_db"] = measured_snr
        r["duration_seconds"] = clean.size / SAMPLE_RATE
    require(len(list((OUTPUT_DIR / FOLDERS[split]).glob("*.wav"))) == 2 * COUNTS[split],
            f"Wrong file count in {split}")
    write_rows(OUTPUT_DIR / f"{split}_metadata.csv", rows)
    print(split, "pairs:", len(rows), "SNR counts:", dict(Counter(r["snr_db"] for r in rows)))

# Only a completely generated and checked dataset receives this marker.
(OUTPUT_DIR / "COMPLETE.json").write_text(json.dumps({"counts": COUNTS, "checks_passed": True}))
print("All splits generated and checked:", OUTPUT_DIR)


In [ ]:
import tarfile

require((OUTPUT_DIR / "COMPLETE.json").is_file(), "Dataset generation is incomplete")
# Save directly to Drive so the archive survives a Colab runtime reset.
archive_path = Path("/content/drive/MyDrive/Denoiser") / f"{OUTPUT_DIR.name}.tar"
require(not archive_path.exists(), f"Archive already exists: {archive_path}; choose a new name")
with tarfile.open(archive_path, "w") as archive:
    archive.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name)
print("Saved:", archive_path)
